# Circuit 1 — State Distillation (5-to-1)

**What it does:** Prepares 5 noisy magic states |T⟩ = T·H|0⟩ on data qubits q0–q4, fans
parity into ancilla q5 via CNOTs, measures q5 mid-circuit, and applies a conditional
X correction on q0 if a syndrome error is detected.

**Notable:** Only circuit in the demo with SPAM errors (`include_spam=True`).

**Two-circuit approach:** `ManyShotRunner` uses Stim (Clifford-only). T gates are
non-Clifford, so the heatmap/GIF pass uses a Clifford stand-in (S replaces T — same
circuit topology). The purity pass uses the real T-gate circuit via `TrajectoryBackend`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend, ManyShotRunner
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

N_SHOTS      = 2000
N_SHOTS_TRAJ = 400

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
def run_purity_pass(circuit, noise_config_twirl, n_qubits, label=""):
    result_traj = TrajectoryBackend().run(
        circuit, noise_model=noise_config_twirl, n_shots=N_SHOTS_TRAJ, seed=42,
    )
    rho = result_traj.final_state
    purities = per_qubit_purities(rho, n_qubits)
    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {'█' * int(p * 20)}")
    print(f"{'─'*50}\n")
    return rho, purities


def visualize_circuit(circuit, result_many, noise_pauli, rho, purities, label, gif_name):
    fig = plot_error_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, circuit.n_qubits)
    plt.show()

    interactive_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    # Use the existing aggregate result for the GIF so the annotation box
    # reads "Aggregate stats (2000 shots)" instead of "Shot stats (single-shot)".
    viz = Visualizer(circuit)
    viz.many_shot_result = result_many   # reuse — no extra simulation needed
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers ready.")

In [ ]:
N_DIST = 6  # q0..q4 = data, q5 = ancilla

def build_distillation_clifford() -> nq.Circuit:
    """Clifford stand-in for ManyShotRunner heatmap/GIF: S replaces T (same topology)."""
    c = nq.Circuit(n_qubits=N_DIST, name="distillation_clifford")
    syndrome = c.add_classical_register("syndrome", 1)
    for q in range(5):
        c.h(q)
        c.s(q)
    for q in range(5):
        c.cnot(q, 5)
    c.measure(5, syndrome[0])
    c.c_if(syndrome[0], value=1).x(0)
    return c


def build_distillation_circuit() -> nq.Circuit:
    """Real distillation circuit with T gates for TrajectoryBackend purity pass."""
    c = nq.Circuit(n_qubits=N_DIST, name="distillation_5to1")
    syndrome = c.add_classical_register("syndrome", 1)
    for q in range(5):
        c.h(q)
        c.tgate(q)
    for q in range(5):
        c.cnot(q, 5)
    c.measure(5, syndrome[0])
    c.c_if(syndrome[0], value=1).x(0)
    return c


# Fill idle slots so wire halos appear in the heatmap
circuit_dist_clifford = fill_idle_with_identities(
    build_distillation_clifford(), gate_times
)
circuit_dist = fill_idle_with_identities(
    build_distillation_circuit(), gate_times
)
print(f"Clifford circuit ops: {len(circuit_dist_clifford.operations)}")
print(f"T-gate circuit ops:   {len(circuit_dist.operations)}")

In [ ]:
noise_dist_clifford = profile.to_noise_model(
    circuit_dist_clifford, mode="t2", representation="pauli_twirl", include_spam=True,
)
noise_dist = profile.to_noise_model(
    circuit_dist, mode="t2", representation="pauli_twirl", include_spam=True,
)

result_dist = ManyShotRunner().run(
    circuit_dist_clifford, n_shots=N_SHOTS, noise_config=noise_dist_clifford, seed=42,
)
print(f"ManyShotRunner done  zero-error fraction: {result_dist.zero_error_fraction:.4f}")

rho_dist, pur_dist = run_purity_pass(circuit_dist, noise_dist, N_DIST, "State Distillation")

In [ ]:
visualize_circuit(
    circuit_dist_clifford, result_dist, noise_dist_clifford, rho_dist, pur_dist,
    label="State Distillation (5-to-1, SPAM on)",
    gif_name="distillation",
)